# Adaptación de dominio de DisorBERT al corpus trans (MLM)

Este cuaderno continúa el entrenamiento de `citiusLTL/DisorBERT` mediante *Masked Language Modeling* (MLM) sobre publicaciones de comunidades trans en Reddit. DisorBERT ya viene adaptado de texto general a salud mental; aquí hacemos la tercera adaptación de dominio (hacia `DysphorBERT`).

La configuración por defecto usa la versión mejorada **`LEXICO_MODO = "depresion+trans"`** con **`PROB_LEX_DEP = 0.5`** (el modelo ve los términos de depresión sin tapar la mitad de las veces) y **`SPLIT_POR_AUTOR = True`** (validación agrupada por autor), escribiendo en `artefactos/disorbert_trans_depresion+trans_pdep50_autor`. La corrida original (`PROB_LEX_DEP = 1.0`, split por publicación en `artefactos/disorbert_trans_depresion+trans`) permanece intacta y se compara automáticamente en la Sección 7.

Entradas esperadas desde `03_adaptacion_mlm/`:
- `artefactos/mlm_corpus.parquet`
- `artefactos/lexico_trans.txt`
- `../02_dataset_gold/datos/gold_1400.csv`
- `../01_datos_reddit/datos/bdi_total.parquet`

## 0. Entorno y configuración

Por defecto entrena la configuración mejorada (`LEXICO_MODO = "depresion+trans"`, `PROB_LEX_DEP = 0.5`, `SPLIT_POR_AUTOR = True`) en una carpeta propia sin tocar la corrida original (`pdep 1.0`, split por post). Si quieres correr la ablación de control, cambia `LEXICO_MODO = "ninguno"`; si ya existe `modelo_final/` para esa etiqueta y `ENTRENAR_SI_EXISTE_MODELO_FINAL = False`, el entrenamiento se salta y pasa directo a evaluar o comparar.

In [30]:
import gc
import hashlib
import html
import itertools
import json
import math
import os
import random
import re
import sys
import time
import unicodedata
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "true"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm
from datasets import Dataset, DatasetDict, Features, List, Value, load_from_disk
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    pipeline,
)


In [31]:
# ── Rutas y control de corrida ─────────────────────────────────────────────
ARTEFACTOS = Path("artefactos")
CORPUS = ARTEFACTOS / "mlm_corpus.parquet"
GOLD_CSV = Path("../02_dataset_gold/datos/gold_1400.csv")
BDI_TOTAL = Path("../01_datos_reddit/datos/bdi_total.parquet")

MODELO_BASE = "citiusLTL/DisorBERT"
LEXICO_MODO = "ninguno"   # "depresion+trans" | "depresion" | "ninguno"
ENTRENAR_SI_EXISTE_MODELO_FINAL = False

# ── Corpus y filtrado ──────────────────────────────────────────────────────
MAX_DOCS = None
MIN_CHARS = 120
DEV = 0.005
EXCLUIR_AUTORES_GOLD = True
SPLIT_POR_AUTOR = True

# ── Tokenización y enmascarado ─────────────────────────────────────────────
MAX_LEN = 256
MLM_PROB = 0.20
PROB_LEX_DEP = 0.5    # La corrida original usó 1.0; 0.5 deja ver términos de depresión sin tapar la mitad de las veces
PROB_LEX_TRANS = 0.7
MAX_MASCARA = 0.40

ETIQUETA_CORRIDA = (
    f"{LEXICO_MODO}_pdep{int(round(PROB_LEX_DEP * 100))}"
    if LEXICO_MODO != "ninguno" else "ninguno"
) + ("_autor" if SPLIT_POR_AUTOR else "_post")
SALIDA = ARTEFACTOS / f"disorbert_trans_{ETIQUETA_CORRIDA}"

# ── Entrenamiento ──────────────────────────────────────────────────────────
EPOCHS = 1
LR = 3e-5
BATCH = 32
GRAD_ACCUM = 2        # lote efectivo = 64
BATCH_EVAL = 32
EVAL_STEPS = 1000
WARMUP = 0.06
WD = 0.01
SEED = 42
BF16 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
SALIDA.mkdir(parents=True, exist_ok=True)

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| rocm/hip:", getattr(torch.version, "hip", None))
print("gpu         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("corrida     :", ETIQUETA_CORRIDA, "->", SALIDA)

python      : 3.12.16
torch       : 2.13.0+rocm10.0.0 | rocm/hip: 7.15.26333
gpu         : AMD Radeon RX 9070 XT
corrida     : ninguno_autor -> artefactos/disorbert_trans_ninguno_autor


## 1. Corpus y control anti-fuga

Limpia URLs, menciones `r/` o `u/`, marcas Markdown y textos cortos (`< 120` caracteres, `< 60` letras u `< 8` palabras). Después elimina cualquier publicación que coincida con el gold por `id`, por hash SHA-1 de texto limpio o por autor (`EXCLUIR_AUTORES_GOLD = True`).

In [32]:
assert CORPUS.exists(), f"No encuentro {CORPUS}"
assert GOLD_CSV.exists(), f"No encuentro {GOLD_CSV}"

# La clave depende de los IDs del gold (no del tamaño del CSV), así que etiquetar con revisar.py no invalida la caché
ids_gold_clave = sorted(pd.read_csv(GOLD_CSV, encoding="utf-8-sig", dtype={"id": str}, usecols=["id"])["id"].astype(str))
clave_corpus = hashlib.sha1(json.dumps({
    "min_chars": MIN_CHARS,
    "max_docs": MAX_DOCS,
    "seed": SEED,
    "excluir_autores": EXCLUIR_AUTORES_GOLD,
    "gold_ids": hashlib.sha1("\n".join(ids_gold_clave).encode("utf-8")).hexdigest(),
    "corpus_bytes": CORPUS.stat().st_size,
}, sort_keys=True).encode("utf-8")).hexdigest()[:12]
RUTA_CORPUS_CACHE = ARTEFACTOS / f"corpus_limpio_cache_{clave_corpus}.parquet"

if RUTA_CORPUS_CACHE.exists():
    corpus = pd.read_parquet(RUTA_CORPUS_CACHE)
    print(f"Cargado corpus limpio en caché ({RUTA_CORPUS_CACHE.name}): {len(corpus):,} posts ({corpus['autor'].nunique():,} autores)")
else:
    t = pq.read_table(CORPUS, columns=["id", "title", "text"]).to_pandas()
    print(f"Filas iniciales en parquet: {len(t):,}")
    t["id"] = t["id"].astype(str)

    RE_URL = re.compile(r"https?://\S+|www\.\S+")
    RE_SUB = re.compile(r"\br/\w+|\bu/\w+", re.I)
    RE_MD = re.compile(r"[*`~]{2,}|_{2,}")
    RE_CTRL = re.compile(r"[\u200b-\u200f\u202a-\u202e\ufeff]")
    RE_ESP = re.compile(r"\s+")


    def limpiar_texto(s):
        s = unicodedata.normalize("NFKC", s)
        s = html.unescape(s)
        s = RE_URL.sub(" ", s)
        s = RE_SUB.sub(" ", s)
        s = s.replace("[deleted]", " ").replace("[removed]", " ")
        s = RE_MD.sub(" ", s)
        s = RE_CTRL.sub(" ", s)
        return RE_ESP.sub(" ", s).strip()


    def sha1_texto(s):
        return hashlib.sha1(s.encode("utf-8")).hexdigest()


    crudos = (t["title"].fillna("").astype(str) + " " + t["text"].fillna("").astype(str)).tolist()
    t["texto"] = [limpiar_texto(s) for s in tqdm(crudos, desc="Limpiando textos", unit="post", mininterval=0.5)]
    del crudos
    n_palabras = t["texto"].str.count(r"\S+")
    t = t[
        (t["texto"].str.len() >= MIN_CHARS)
        & (t["texto"].str.count(r"[A-Za-z]") >= 60)
        & (n_palabras >= 8)
    ].drop_duplicates("texto")
    print(f"Tras limpieza y deduplicación: {len(t):,}")

    # Control anti-fuga contra gold_1400.csv (ID, hash de texto y autores)
    g = pd.read_csv(GOLD_CSV, encoding="utf-8-sig", dtype={"id": str})
    ids_gold = set(g["id"].astype(str))
    hashes_gold = set((g["titulo"].fillna("") + " " + g["texto"].fillna("")).map(limpiar_texto).map(sha1_texto))

    corpus = t[["id", "texto"]].reset_index(drop=True)
    corpus["hash"] = [sha1_texto(s) for s in tqdm(corpus["texto"].tolist(), desc="Calculando SHA-1", unit="post", mininterval=0.5)]
    corpus = corpus[~(corpus["id"].isin(ids_gold) | corpus["hash"].isin(hashes_gold))].copy()
    print(f"Tras quitar colisiones ID/hash con el gold: {len(corpus):,}")

    AUTORES_ANON = {
        "[deleted]", "[removed]", "deleted", "removed",
        "AutoModerator", "automoderator", "None", "nan", "",
    }
    if BDI_TOTAL.exists():
        au = pq.read_table(BDI_TOTAL, columns=["id", "author"]).to_pandas()
        au["id"] = au["id"].astype(str)
        au["author"] = au["author"].fillna("").astype(str)
        autores_gold = set(au.loc[au["id"].isin(ids_gold), "author"]) - AUTORES_ANON
        ids_autores = set(au.loc[au["author"].isin(autores_gold), "id"])
        n_autores = int(corpus["id"].isin(ids_autores).sum())
        print(f"Posts de los {len(autores_gold):,} autores del gold en corpus: {n_autores:,}")
        if EXCLUIR_AUTORES_GOLD:
            corpus = corpus[~corpus["id"].isin(ids_autores)].copy()
            print(f"Tras exclusión por autor: {len(corpus):,}")
        id_a_autor = dict(zip(au["id"], au["author"]))
    else:
        id_a_autor = {}

    corpus["autor"] = [
        f"anon_{pid}" if id_a_autor.get(pid, "") in AUTORES_ANON else id_a_autor[pid]
        for pid in corpus["id"]
    ]

    assert not corpus["id"].isin(ids_gold).any() and not corpus["hash"].isin(hashes_gold).any()
    corpus = corpus.drop(columns=["hash"]).reset_index(drop=True)
    if MAX_DOCS is not None and len(corpus) > MAX_DOCS:
        corpus = corpus.sample(n=MAX_DOCS, random_state=SEED).reset_index(drop=True)

    corpus.to_parquet(RUTA_CORPUS_CACHE, index=False)
    print(f"Corpus final listo y guardado en caché ({RUTA_CORPUS_CACHE.name}): {len(corpus):,} posts ({corpus['autor'].nunique():,} autores)")


Cargado corpus limpio en caché (corpus_limpio_cache_8d6078d13aac.parquet): 1,441,325 posts (514,458 autores)


## 2. Tokenizador, léxicos y `CollatorGuiado`

Carga el tokenizador WordPiece de DisorBERT, arma los dos léxicos (126 palabras de depresión y 274 términos trans ampliados con `artefactos/lexico_trans.txt`, generado con *keyness* en `artefactos/construir_lexico_trans.py`) y define `CollatorGuiado` para enmascarar palabras completas al vuelo.

In [33]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE, use_fast=True)
assert getattr(tokenizer, "is_fast", False), "Se requiere tokenizador rápido para mapear palabras."

LEXICO_DEPRESION = (
    "abandon abandoned abused afraid agony alone angry anguish anxiety anxious apathy ashamed "
    "awful bad betrayed bitter blame blamed blue broken burden cry crying dead death defeated "
    "depressed depressing depression despair desperate destroyed die died disappointment "
    "disgusted dislike dread dysphoria dysphoric dysmorphia empty enraged exhausted failure "
    "failed failing fake fear fearful flashback friendless frustrated furious grief guilty hate "
    "hated helpless hopeless hopelessness hurt ignored inferior insomnia invisible irritable "
    "jealous joyless kill lonely lose loser lost low meaningless misgendered misgendering "
    "miserable mood numb obsessed overwhelmed pain painful panic paranoid pathetic pointless "
    "punished regret rejected restless ruined sad sadness scared selfharm shame shamed sick "
    "silence sleepless stressed stupid suffer suffering suicide suicidal tired trapped trash "
    "transphobia transphobic ugly uncomfortable unhappy unwanted useless victim weak weep "
    "weird worthless worthlessness wrong"
).split()

LEXICO_TRANS = (
    "hrt t4t deadname deadnamed deadnaming euphoria binder binding packing packer tucking "
    "stealth clocked clock chaser boymode girlmode transmasc transfem nonbinary enby afab "
    "amab cis cisgender ftm mtf testosterone estrogen estradiol spironolactone spiro blockers "
    "transition transitioning detransition misgender pronouns passing egg ffs gatekeeping "
    "cyproterone cypro cypionate bicalutamide finasteride valerate androgen androgens blocker "
    "sublingual sublingually monotherapy regimen endocrinologist endo libido fertility infertile "
    "puberty dosage dosages dose doses hormone hormones hormonal pmol nmol redistribution "
    "feminizing genderfluid androgynous androgyny femme butch tomboy masc gnc agab "
    "transmasculine transness gendered femininity masculinity detransitioning detransitioned "
    "transitioned transitions presenting hetero het pronoun queer binders bind underworks "
    "gc2b limpy bras boobs diy"
).split()

ruta_trans = ARTEFACTOS / "lexico_trans.txt"
if ruta_trans.exists():
    LEXICO_TRANS += [p.strip().lower() for p in ruta_trans.read_text(encoding="utf-8").splitlines() if p.strip()]

LEXICO_TRANS = sorted(set(LEXICO_TRANS))
LEXICO_DEPRESION = sorted(set(LEXICO_DEPRESION) - set(LEXICO_TRANS))
SET_DEP, SET_TRANS = set(LEXICO_DEPRESION), set(LEXICO_TRANS)

HELDOUT = {"genderqueer", "agender", "progesterone", "transmisogyny", "mastectomy", "crossdresser"}
assert not (HELDOUT & (SET_DEP | SET_TRANS)), "Las palabras held-out no pueden estar en los léxicos."
assert len(LEXICO_DEPRESION) == 126 and len(LEXICO_TRANS) == 274

print(f"Vocabulario: {len(tokenizer):,} · Depresión: {len(LEXICO_DEPRESION)} · Trans: {len(LEXICO_TRANS)}")
print("Ejemplo WordPiece ->", {w: tokenizer.tokenize(w) for w in ("hrt", "t4t", "dysphoria", "deadname")})

Vocabulario: 30,522 · Depresión: 126 · Trans: 274
Ejemplo WordPiece -> {'hrt': ['hr', '##t'], 't4t': ['t', '##4', '##t'], 'dysphoria': ['d', '##ys', '##ph', '##oria'], 'deadname': ['dead', '##name']}


In [34]:
class CollatorGuiado:
    """Enmascara palabras enteras de léxico (todas sus piezas ## juntas) y tokens
    sueltos al azar en el resto. En modo 'ninguno' aplica MLM aleatorio estándar."""

    def __init__(self, tok, mlm_prob=0.20, modo="depresion+trans",
                 prob_lex_dep=1.0, prob_lex_trans=0.7, max_mascara=0.40,
                 generador=None, semilla=42):
        self.tok = tok
        self.p = mlm_prob
        self.modo = modo
        self.p_dep = prob_lex_dep
        self.p_trans = prob_lex_trans
        self.tope = max_mascara
        self.generador = generador
        self.semilla = semilla
        self.evaluando = False
        self.ids_especiales = torch.tensor(sorted(tok.all_special_ids), dtype=torch.long)

    def reiniciar(self):
        if self.generador is not None:
            self.generador.manual_seed(self.semilla)

    def _gen(self):
        return self.generador if (self.evaluando and self.generador is not None) else None

    def _prob_de_palabra(self, clase):
        if clase == 1 and self.modo in ("depresion", "depresion+trans"):
            return self.p_dep
        if clase == 2 and self.modo == "depresion+trans":
            return self.p_trans
        return None

    def _enmascarar_lexico(self, features, mascara):
        grupos_por_fila = []
        posiciones_lex = torch.zeros_like(mascara)
        for fila, feat in enumerate(features):
            clases, pids = feat.get("lex_clase"), feat.get("palabra_id")
            grupos, previo = [], None
            if clases and pids:
                for i, (c, p) in enumerate(zip(clases, pids)):
                    if p < 0 or self._prob_de_palabra(c) is None:
                        continue
                    posiciones_lex[fila, i] = True
                    if previo and previo[0] == p and previo[1] == i - 1:
                        grupos[-1].append(i)
                    else:
                        grupos.append([i])
                    previo = (p, i)
                if grupos:
                    azar_pal = torch.rand(len(grupos), generator=self._gen()).tolist()
                    for posis, r in zip(grupos, azar_pal):
                        if r < self._prob_de_palabra(clases[posis[0]]):
                            mascara[fila, posis] = True
            grupos_por_fila.append(grupos)
        return grupos_por_fila, posiciones_lex

    def _recortar_y_asegurar(self, mascara, atencion, especiales, grupos_por_fila, posiciones_lex):
        validos = atencion.sum(dim=1)
        for fila in range(mascara.shape[0]):
            tope_f = max(1, int(self.tope * int(validos[fila])))
            sobra = int(mascara[fila].sum()) - tope_f
            if sobra > 0:
                unidades = [g for g in grupos_por_fila[fila] if all(bool(mascara[fila, j]) for j in g)]
                unidades += [[i] for i in mascara[fila].nonzero(as_tuple=True)[0].tolist()
                             if not bool(posiciones_lex[fila, i])]
                for u in torch.randperm(len(unidades), generator=self._gen()).tolist():
                    if sobra <= 0:
                        break
                    mascara[fila, unidades[u]] = False
                    sobra -= len(unidades[u])

            if int(mascara[fila].sum()) == 0 and int(validos[fila]) > 2:
                libre = (~(especiales[fila] | posiciones_lex[fila])).nonzero(as_tuple=True)[0]
                if len(libre) == 0:
                    libre = (~especiales[fila]).nonzero(as_tuple=True)[0]
                if len(libre) == 0:
                    continue
                idx = int(torch.randint(len(libre), (1,), generator=self._gen()))
                mascara[fila, libre[idx]] = True

    def _aplicar_80_10_10(self, entrada, mascara):
        r1 = torch.rand(entrada.shape, generator=self._gen())
        r2 = torch.rand(entrada.shape, generator=self._gen())
        azar = (r1 < 0.1) & mascara
        mantener = ~(r1 < 0.1) & (r2 < 0.1) & mascara
        entrada[mascara & ~azar & ~mantener] = self.tok.mask_token_id
        if azar.any():
            entrada[azar] = torch.randint(len(self.tok), (int(azar.sum()),), generator=self._gen())

    def __call__(self, features):
        batch = self.tok.pad([{"input_ids": f["input_ids"]} for f in features], return_tensors="pt")
        entrada = batch["input_ids"].clone()
        etiquetas = entrada.clone()
        atencion = batch["attention_mask"]
        especiales = torch.isin(entrada, self.ids_especiales)
        mascara = torch.zeros_like(entrada, dtype=torch.bool)

        grupos_por_fila, posiciones_lex = self._enmascarar_lexico(features, mascara)
        prob_azar = torch.full(entrada.shape, self.p)
        prob_azar.masked_fill_(especiales | posiciones_lex, 0.0)
        mascara |= torch.rand(entrada.shape, generator=self._gen()) < prob_azar

        self._recortar_y_asegurar(mascara, atencion, especiales, grupos_por_fila, posiciones_lex)
        self._aplicar_80_10_10(entrada, mascara)
        etiquetas[~mascara] = -100
        return {"input_ids": entrada, "attention_mask": atencion, "labels": etiquetas}


colador = CollatorGuiado(
    tokenizer, MLM_PROB, LEXICO_MODO, PROB_LEX_DEP, PROB_LEX_TRANS, MAX_MASCARA,
    generador=torch.Generator().manual_seed(SEED + 1), semilla=SEED + 1,
)
colador_neutral = CollatorGuiado(
    tokenizer, 0.15, "ninguno", 0.0, 0.0, MAX_MASCARA,
    generador=torch.Generator().manual_seed(SEED + 7), semilla=SEED + 7,
)
print("Collator principal:", LEXICO_MODO, "| Collator neutral: aleatorio 15 %")


Collator principal: ninguno | Collator neutral: aleatorio 15 %


## 3. Empaquetado en bloques de 256 tokens

Separa validación (`DEV = 0.005`) antes de empaquetar. Con `SPLIT_POR_AUTOR = True` (default) agrupa todas las publicaciones de cada autor en el mismo lado del corte para evitar fugas en la perplejidad de validación; con `False` usa split aleatorio por publicación. Luego concatena los textos tokenizados en bloques de 256 tokens sin padding y corre un chequeo rápido del collator.

In [35]:

if SPLIT_POR_AUTOR:
    grupos = {}
    for texto, autor in zip(corpus["texto"], corpus["autor"]):
        grupos.setdefault(autor, []).append(texto)
    autores = sorted(grupos)
    random.Random(SEED).shuffle(autores)
    meta_val = round(len(corpus) * DEV)
    textos_val, textos_train = [], []
    autores_val, autores_train = set(), set()
    for autor in autores:
        if len(textos_val) < meta_val:
            textos_val.extend(grupos[autor])
            autores_val.add(autor)
        else:
            textos_train.extend(grupos[autor])
            autores_train.add(autor)
    assert not (autores_train & autores_val), "Fuga de autores entre train y validation."
    ds = DatasetDict({
        "train": Dataset.from_dict({"text": textos_train}),
        "validation": Dataset.from_dict({"text": textos_val}),
    })
    n_aut_train, n_aut_val = len(autores_train), len(autores_val)
    del grupos, autores, textos_train, textos_val, autores_train, autores_val
    gc.collect()
else:
    split = Dataset.from_dict({"text": corpus["texto"].tolist(), "autor": corpus["autor"].tolist()}).train_test_split(
        test_size=DEV, seed=SEED
    )
    n_aut_train = len(set(split["train"]["autor"]))
    n_aut_val = len(set(split["test"]["autor"]))
    ds = DatasetDict({
        "train": split["train"].remove_columns("autor"),
        "validation": split["test"].remove_columns("autor"),
    })

print(f"Split ({'por autor' if SPLIT_POR_AUTOR else 'por post'}): "
      f"train={len(ds['train']):,} posts ({n_aut_train:,} autores) | "
      f"val={len(ds['validation']):,} posts ({n_aut_val:,} autores)")

clave_bloques = hashlib.sha1(json.dumps({
    "corpus": clave_corpus,
    "lex_dep": hashlib.sha1("\n".join(LEXICO_DEPRESION).encode("utf-8")).hexdigest(),
    "lex_trans": hashlib.sha1("\n".join(LEXICO_TRANS).encode("utf-8")).hexdigest(),
    "modelo": MODELO_BASE,
    "max_len": MAX_LEN,
    "min_chars": MIN_CHARS,
    "max_docs": MAX_DOCS,
    "dev": DEV,
    "seed": SEED,
    "excluir_autores": EXCLUIR_AUTORES_GOLD,
    "split_por_autor": SPLIT_POR_AUTOR,
    "n_train": len(ds["train"]),
    "n_val": len(ds["validation"]),
    "n_dep": len(LEXICO_DEPRESION),
    "n_trans": len(LEXICO_TRANS),
}, sort_keys=True).encode("utf-8")).hexdigest()[:12]
# Se guarda en el disco nativo de WSL (ext4): el entrenamiento lee bloques al azar y hacerlo sobre /mnt/d es muy lento
DIR_CACHE_LOCAL = Path.home() / ".cache" / "dysphorbert"
DIR_CACHE_LOCAL.mkdir(parents=True, exist_ok=True)
RUTA_CACHE_BLOQUES = DIR_CACHE_LOCAL / f"bloques_cache_{clave_bloques}"

# Tipos compactos: ~2.6 GB en vez de ~7 GB con int64 (vocabulario de 30,522 cabe en int32; clases 0/1/2 en int8)
FEATURES_BLOQUES = Features({
    "input_ids": List(Value("int32")),
    "lex_clase": List(Value("int8")),
    "palabra_id": List(Value("int32")),
})

PUNT_STRIP = ".,;:!?()[]\"'"


def tokenizar_y_marcar(lote):
    encs = tokenizer(lote["text"], truncation=False, return_offsets_mapping=True)
    lex_clases, pal_ids = [], []
    for i, texto in enumerate(lote["text"]):
        wids = encs.word_ids(i)
        offs = encs["offset_mapping"][i]
        span = {}
        for wid, (s, e) in zip(wids, offs):
            if wid is not None:
                if wid in span:
                    span[wid][1] = e
                else:
                    span[wid] = [s, e]
        palabras_lex = {}
        for wid, (s, e) in span.items():
            pal = texto[s:e].lower().strip(PUNT_STRIP)
            if pal in SET_DEP:
                palabras_lex[wid] = 1
            elif pal in SET_TRANS:
                palabras_lex[wid] = 2
        lex_clases.append([palabras_lex.get(w, 0) for w in wids])
        pal_ids.append([w if w in palabras_lex else -1 for w in wids])
    return {"input_ids": encs["input_ids"], "lex_clase": lex_clases, "palabra_id": pal_ids}


def agrupar_en_bloques(lote):
    union = {k: list(itertools.chain.from_iterable(lote[k])) for k in lote}
    n = (len(union["input_ids"]) // MAX_LEN) * MAX_LEN
    return {k: [union[k][i:i + MAX_LEN] for i in range(0, n, MAX_LEN)] for k in union}


if RUTA_CACHE_BLOQUES.exists():
    print(f"Cargando bloques en caché desde {RUTA_CACHE_BLOQUES} ...")
    bloques = load_from_disk(str(RUTA_CACHE_BLOQUES))
else:
    print(f"Tokenizando y empaquetando en 1 proceso con Rust paralelo (se guardará en {RUTA_CACHE_BLOQUES}) ...")
    tokenizadas = ds.map(
        tokenizar_y_marcar,
        batched=True,
        batch_size=2000,
        writer_batch_size=2000,
        remove_columns=["text"],
        features=FEATURES_BLOQUES,
        desc="Tokenizando",
    )
    bloques = tokenizadas.map(
        agrupar_en_bloques,
        batched=True,
        batch_size=2000,
        writer_batch_size=2000,
        features=FEATURES_BLOQUES,
        desc="Empaquetando",
    )
    bloques.save_to_disk(str(RUTA_CACHE_BLOQUES))

pasos = math.ceil(len(bloques["train"]) / (BATCH * GRAD_ACCUM))
print(f"Train: {len(bloques['train']):,} bloques ({len(bloques['train']) * MAX_LEN:,} tokens)")
print(f"Valid: {len(bloques['validation']):,} bloques · Pasos por época: {pasos:,}")

# Chequeo rápido del collator sobre 200 bloques
muestra = [bloques["train"][i] for i in range(200)]
colador.evaluando = False
lote = colador(muestra)
enm = lote["labels"] != -100
frac = enm.sum(1) / lote["attention_mask"].sum(1).clamp(min=1)
assert float(frac.max()) <= MAX_MASCARA + 0.01
colador.evaluando = True
colador.reiniciar()
l1 = colador(muestra)
colador.reiniciar()
l2 = colador(muestra)
assert torch.equal(l1["labels"], l2["labels"])
colador.evaluando = False
print(f"Collator OK · % enmascarado real en 200 bloques: {100 * float(enm.sum()) / int(lote['attention_mask'].sum()):.1f}%")


Split (por autor): train=1,434,118 posts (511,661 autores) | val=7,207 posts (2,797 autores)
Cargando bloques en caché desde /home/lucy/.cache/dysphorbert/bloques_cache_f9a7fbeae4c1 ...
Train: 1,162,051 bloques (297,485,056 tokens)
Valid: 5,848 bloques · Pasos por época: 18,158
Collator OK · % enmascarado real en 200 bloques: 19.7%


## 4. Línea base (perplejidad y 16 sondas)

Antes de modificar los pesos, mide la perplejidad de `DisorBERT` base con máscaras fijas (modo actual y neutral 15 %) y evalúa 16 frases *cloze*: 10 palabras que aparecen en el léxico y 6 palabras *held-out* que están fuera de ambos léxicos para medir generalización real.

In [36]:
M_MASK = tokenizer.mask_token

SONDAS = [
    (f"I have been on {M_MASK} for eight months and my voice is finally dropping.", "testosterone"),
    (f"For the first time in years I went full {M_MASK} at work and nobody noticed.", "girlmode"),
    (f"I am looking for {M_MASK} because I am tired of dating people who do not get it.", "t4t"),
    (f"She asked for my old name, my {M_MASK}, and I just froze.", "deadname"),
    (f"My {M_MASK} gets so bad some nights that I cannot get out of bed.", "dysphoria"),
    (f"The first time a stranger used my new name I felt pure {M_MASK}.", "euphoria"),
    (f"He only wanted to talk about my body, total {M_MASK} behaviour.", "chaser"),
    (f"I have been {M_MASK} for two years and even my coworkers do not know.", "stealth"),
    (f"I keep worrying that people can {M_MASK} me from my voice alone.", "clock"),
    (f"I started {M_MASK} last month and my chest finally feels right.", "binding"),
    # 6 sondas fuera del léxico (held-out)
    (f"I identify as {M_MASK} because I do not feel like a man or a woman.", "genderqueer"),
    (f"I have never felt like a man or a woman and I use the word {M_MASK} for myself.", "agender"),
    (f"My endocrinologist wants to add {M_MASK} to my regimen next month.", "progesterone"),
    (f"The {M_MASK} in this community is exhausting and nobody wants to talk about it.", "transmisogyny"),
    (f"After my {M_MASK} I finally felt comfortable going shirtless at the pool.", "mastectomy"),
    (f"I am not trans, I am just a {M_MASK} who likes wearing dresses at home.", "crossdresser"),
]


def calcular_perplejidad(modelo_eval, dataset, colador_eval=colador, max_lotes=200):
    modelo_eval.eval()
    colador_eval.evaluando = True
    colador_eval.reiniciar()
    cargador = torch.utils.data.DataLoader(dataset, batch_size=BATCH_EVAL, collate_fn=colador_eval)
    perdidas = []
    for i, lote_eval in enumerate(cargador):
        if i >= max_lotes:
            break
        lote_eval = {k: v.to(DISPOSITIVO) for k, v in lote_eval.items()}
        with torch.no_grad():
            perdidas.append(float(modelo_eval(**lote_eval).loss))
    colador_eval.evaluando = False
    return math.exp(sum(perdidas) / len(perdidas))


def puntuar_sonda(modelo_eval, plantilla, palabra):
    prefijo, _, sufijo = plantilla.partition(M_MASK)
    ids_pre = tokenizer(prefijo, add_special_tokens=False)["input_ids"]
    ids_obj = tokenizer(palabra, add_special_tokens=False)["input_ids"]
    ids_suf = tokenizer(sufijo, add_special_tokens=False)["input_ids"]
    ids = [tokenizer.cls_token_id] + ids_pre + [tokenizer.mask_token_id] * len(ids_obj) + ids_suf + [tokenizer.sep_token_id]
    tensor = torch.tensor([ids], device=DISPOSITIVO)
    modelo_eval.eval()
    with torch.no_grad():
        logprobs = modelo_eval(input_ids=tensor, attention_mask=torch.ones_like(tensor)).logits[0].log_softmax(-1)
    inicio = 1 + len(ids_pre)
    lp, rangos = 0.0, []
    for k, tid in enumerate(ids_obj):
        fila = logprobs[inicio + k]
        lp += float(fila[tid])
        rangos.append(int((fila > fila[tid]).sum()) + 1)
    return lp / len(ids_obj), sum(rangos) / len(rangos)


modelo = AutoModelForMaskedLM.from_pretrained(MODELO_BASE).to(DISPOSITIVO)
PPL_BASE = calcular_perplejidad(modelo, bloques["validation"], colador)
PPL_BASE_NEUTRAL = calcular_perplejidad(modelo, bloques["validation"], colador_neutral)
SONDAS_BASE = {pal: puntuar_sonda(modelo, tpl, pal) for tpl, pal in SONDAS}

print(f"Base -> PPL ({LEXICO_MODO}): {PPL_BASE:.3f} | PPL neutral 15 %: {PPL_BASE_NEUTRAL:.3f}")

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

Base -> PPL (ninguno): 45.560 | PPL neutral 15 %: 35.536


## 5. Entrenamiento MLM reanudable

Usa `huella.json` completa para impedir mezclar checkpoints de otra configuración. Si interrumpes el kernel durante el entrenamiento y vuelves a correr la celda, detecta automáticamente el último `checkpoint-*` en `SALIDA/checkpoints/` y continúa desde ahí.

In [37]:
class EntrenadorMLM(Trainer):
    """Fija las máscaras de validación al inicio de cada evaluación (eval en 1 hilo para determinismo)."""

    def get_eval_dataloader(self, eval_dataset=None):
        dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        return torch.utils.data.DataLoader(
            dataset,
            batch_size=self.args.per_device_eval_batch_size,
            collate_fn=self.data_collator,
            num_workers=0,
            pin_memory=self.args.dataloader_pin_memory,
        )

    def evaluate(self, *args, **kwargs):
        if hasattr(self.data_collator, "reiniciar"):
            self.data_collator.evaluando = True
            self.data_collator.reiniciar()
        try:
            return super().evaluate(*args, **kwargs)
        finally:
            self.data_collator.evaluando = False


def sha_lista(lista):
    return hashlib.sha1("\n".join(sorted(lista)).encode("utf-8")).hexdigest()[:12]


usar_bf16 = bool(BF16 and torch.cuda.is_available() and torch.cuda.is_bf16_supported())
usar_fp16 = bool(torch.cuda.is_available() and not usar_bf16)

HUELLA = {
    "modelo": MODELO_BASE,
    "modo": LEXICO_MODO,
    "etiqueta_corrida": ETIQUETA_CORRIDA,
    "lex_dep": sha_lista(LEXICO_DEPRESION),
    "lex_trans": sha_lista(LEXICO_TRANS),
    "mlm_prob": MLM_PROB,
    "p_dep": PROB_LEX_DEP,
    "p_trans": PROB_LEX_TRANS,
    "max_mascara": MAX_MASCARA,
    "max_len": MAX_LEN,
    "max_docs": MAX_DOCS,
    "min_chars": MIN_CHARS,
    "dev": DEV,
    "seed": SEED,
    "lr": LR,
    "epochs": EPOCHS,
    "batch": BATCH,
    "grad_accum": GRAD_ACCUM,
    "lote_efectivo": BATCH * GRAD_ACCUM,
    "eval_steps": EVAL_STEPS,
    "warmup": WARMUP,
    "wd": WD,
    "bf16": usar_bf16,
    "excluir_autores": EXCLUIR_AUTORES_GOLD,
    "split_por_autor": SPLIT_POR_AUTOR,
    "posts_usados": int(len(corpus)),
}

RUTA_MODELO = SALIDA / "modelo_final"
RUTA_HUELLA = SALIDA / "huella.json"
CARPETA_CKPT = SALIDA / "checkpoints"
puntos = sorted(CARPETA_CKPT.glob("checkpoint-*"), key=lambda p: int(p.name.rsplit("-", 1)[-1])) if CARPETA_CKPT.exists() else []
reanudar = puntos[-1] if puntos else None

if reanudar is not None:
    assert RUTA_HUELLA.exists(), f"Hay checkpoints en {CARPETA_CKPT} sin huella.json."
    previa = json.loads(RUTA_HUELLA.read_text(encoding="utf-8"))
    assert previa == HUELLA, f"Huella distinta en {SALIDA}: cambia SALIDA o borra checkpoints."
RUTA_HUELLA.write_text(json.dumps(HUELLA, ensure_ascii=False, indent=2), encoding="utf-8")

entrenado_ahora = False
if (RUTA_MODELO / "model.safetensors").exists() and not ENTRENAR_SI_EXISTE_MODELO_FINAL:
    print(f"Ya existe {RUTA_MODELO}; se omite el entrenamiento.")
    modelo = AutoModelForMaskedLM.from_pretrained(RUTA_MODELO).to(DISPOSITIVO)
else:
    args = TrainingArguments(
        output_dir=str(CARPETA_CKPT),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH_EVAL,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        weight_decay=WD,
        warmup_steps=WARMUP,
        logging_steps=50,
        logging_first_step=True,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=EVAL_STEPS,
        eval_steps=EVAL_STEPS,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=4,
        dataloader_pin_memory=torch.cuda.is_available(),
        dataloader_persistent_workers=True,
        report_to=[],
        remove_unused_columns=False,
        bf16=usar_bf16,
        fp16=usar_fp16,
    )
    trainer = EntrenadorMLM(
        model=modelo,
        args=args,
        train_dataset=bloques["train"],
        eval_dataset=bloques["validation"],
        data_collator=colador,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        processing_class=tokenizer,
    )
    t0 = time.time()
    trainer.train(resume_from_checkpoint=str(reanudar) if reanudar is not None else None)
    print(f"Entrenado en {(time.time() - t0) / 60:.1f} min · paso global {trainer.state.global_step}")
    modelo = trainer.model
    trainer.state.save_to_json(str(SALIDA / "trainer_state.json"))
    entrenado_ahora = True


Step,Training Loss,Validation Loss
1000,4.114189,1.960220
2000,3.922138,1.861949
3000,3.804549,1.812715
4000,3.699278,1.778006
5000,3.692496,1.754818
6000,3.638482,1.735390
7000,3.624673,1.719041
8000,3.603544,1.706549
9000,3.573061,1.696274
10000,3.490023,1.686609


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (12 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


Entrenado en 88.8 min · paso global 18158


## 6. Evaluación, guardado y prueba `fill-mask`

Evalúa el modelo final contra la línea base (perplejidad y 16 sondas), guarda `modelo_final/` junto con `metadatos_adaptacion.json` y muestra cuatro predicciones de ejemplo con `fill-mask`.

In [38]:
PPL_ADAPT = calcular_perplejidad(modelo, bloques["validation"], colador)
PPL_ADAPT_NEUTRAL = calcular_perplejidad(modelo, bloques["validation"], colador_neutral)
print(f"PPL ({LEXICO_MODO}): {PPL_BASE:.3f} -> {PPL_ADAPT:.3f}")
print(f"PPL neutral 15 %    : {PPL_BASE_NEUTRAL:.3f} -> {PPL_ADAPT_NEUTRAL:.3f}")

filas = []
for tpl, pal in SONDAS:
    lp_b, r_b = SONDAS_BASE[pal]
    lp_a, r_a = puntuar_sonda(modelo, tpl, pal)
    filas.append({
        "jerga": pal,
        "grupo": "en léxico" if pal in (SET_TRANS | SET_DEP) else "fuera del léxico",
        "logp_base": round(lp_b, 4),
        "logp_adaptado": round(lp_a, 4),
        "delta": round(lp_a - lp_b, 4),
        "rango_base": round(r_b, 2),
        "rango_adaptado": round(r_a, 2),
    })

tabla_sondas = pd.DataFrame(filas).sort_values("delta", ascending=False).reset_index(drop=True)
print("\nResumen por grupo:")
print(tabla_sondas.groupby("grupo")["delta"].agg(["mean", "count"]).round(3))
print(tabla_sondas.to_string(index=False))

if entrenado_ahora or not (RUTA_MODELO / "metadatos_adaptacion.json").exists():
    modelo.save_pretrained(RUTA_MODELO)
    tokenizer.save_pretrained(RUTA_MODELO)
    estado_ruta = SALIDA / "trainer_state.json"
    estado_info = json.loads(estado_ruta.read_text(encoding="utf-8")) if estado_ruta.exists() else {}
    metadatos = {
        "modelo_base": MODELO_BASE,
        "etiqueta_corrida": ETIQUETA_CORRIDA,
        "fecha": time.strftime("%Y-%m-%d %H:%M"),
        "corpus": str(CORPUS),
        "posts_usados": int(len(corpus)),
        "posts_train": int(len(ds["train"])),
        "posts_val": int(len(ds["validation"])),
        "split_por_autor": SPLIT_POR_AUTOR,
        "tokens_por_epoca": int(len(bloques["train"]) * MAX_LEN),
        "tokens_procesados": int(len(bloques["train"]) * MAX_LEN * EPOCHS),
        "max_len": MAX_LEN,
        "mlm_prob": MLM_PROB,
        "lexico_modo": LEXICO_MODO,
        "lexico_dep_palabras": len(LEXICO_DEPRESION),
        "lexico_trans_palabras": len(LEXICO_TRANS),
        "prob_lex_dep": PROB_LEX_DEP,
        "prob_lex_trans": PROB_LEX_TRANS,
        "max_mascara": MAX_MASCARA,
        "epocas": EPOCHS,
        "learning_rate": LR,
        "batch": BATCH,
        "grad_accum": GRAD_ACCUM,
        "lote_efectivo": BATCH * GRAD_ACCUM,
        "eval_steps": EVAL_STEPS,
        "best_global_step": estado_info.get("best_global_step"),
        "eval_loss_mejor": estado_info.get("best_metric"),
        "perplejidad_base": round(PPL_BASE, 3),
        "perplejidad_adaptada": round(PPL_ADAPT, 3),
        "perplejidad_neutral_base": round(PPL_BASE_NEUTRAL, 3),
        "perplejidad_neutral_adaptada": round(PPL_ADAPT_NEUTRAL, 3),
        "mejora_logp_media": round(float(tabla_sondas["delta"].mean()), 3),
        "mejora_logp_fuera_lexico": round(float(tabla_sondas.loc[tabla_sondas["grupo"] == "fuera del léxico", "delta"].mean()), 3),
        "sondas": filas,
        "dispositivo": DISPOSITIVO,
        "rocm": getattr(torch.version, "hip", None),
        "licencia": "CC BY-NC 4.0",
    }
    (RUTA_MODELO / "metadatos_adaptacion.json").write_text(
        json.dumps(metadatos, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("\nGuardado modelo_final y metadatos_adaptacion.json en:", RUTA_MODELO)

relleno = pipeline("fill-mask", model=str(RUTA_MODELO), tokenizer=str(RUTA_MODELO),
                   device=0 if torch.cuda.is_available() else -1, top_k=5)
for frase in [
    f"I came out as {M_MASK} to my best friend last week.",
    f"I have been on {M_MASK} for a year and my voice finally dropped.",
    f"My {M_MASK} has been so bad I cannot leave the house.",
    f"I feel so {M_MASK} lately and I do not know why.",
]:
    print("\n>>", frase)
    for r in relleno(frase):
        print(f"   {r['score']:.4f}  {r['token_str']:<14} {r['sequence'][:85]}")

PPL (ninguno): 45.560 -> 5.185
PPL neutral 15 %    : 35.536 -> 4.471

Resumen por grupo:
                   mean  count
grupo                         
en léxico         5.108     10
fuera del léxico  0.112      6
        jerga            grupo  logp_base  logp_adaptado   delta  rango_base  rango_adaptado
        clock        en léxico   -15.4822        -2.5096 12.9726     2979.00            4.00
 testosterone        en léxico   -14.0806        -1.7656 12.3150     7609.00            3.00
      stealth        en léxico   -13.8820        -2.5948 11.2873     6842.00            3.00
      binding        en léxico   -10.5269        -1.0648  9.4620      963.00            2.00
     deadname        en léxico    -4.3176        -1.8156  2.5021       86.00            2.00
     girlmode        en léxico    -8.4558        -6.4155  2.0403      950.33           84.00
      agender fuera del léxico   -10.8570        -8.9576  1.8995     3421.00          377.00
 crossdresser fuera del léxico    -7.5420  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Guardado modelo_final y metadatos_adaptacion.json en: artefactos/disorbert_trans_ninguno_autor/modelo_final


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


>> I came out as [MASK] to my best friend last week.
   0.7630  trans          i came out as trans to my best friend last week.
   0.1022  transgender    i came out as transgender to my best friend last week.
   0.0373  gay            i came out as gay to my best friend last week.
   0.0274  bisexual       i came out as bisexual to my best friend last week.
   0.0215  lesbian        i came out as lesbian to my best friend last week.

>> I have been on [MASK] for a year and my voice finally dropped.
   0.6563  t              i have been on t for a year and my voice finally dropped.
   0.1619  testosterone   i have been on testosterone for a year and my voice finally dropped.
   0.1078  hormones       i have been on hormones for a year and my voice finally dropped.
   0.0620  e              i have been on e for a year and my voice finally dropped.
   0.0020  it             i have been on it for a year and my voice finally dropped.

>> My [MASK] has been so bad I cannot leave the house.


## 7. Comparación de corridas y curvas corregidas

Esta sección solo requiere haber corrido las dos primeras celdas (`imports` y `config`). Descubre automáticamente todas las carpetas `artefactos/disorbert_trans_*` con `trainer_state.json` o checkpoints, incluyendo la corrida original (`depresion+trans`), la mejorada (`depresion+trans_pdep50_autor`), la ablación (`ninguno_autor`) y cualquier experimento futuro. Para cada corrida divide `train_loss` entre su propio `grad_accum` (leído de `huella.json` o `metadatos_adaptacion.json`, con respaldo en `GRAD_ACCUM`), imprime la tabla comparativa completa y guarda `artefactos/curvas_mlm.png`.

Nota de licencia: `citiusLTL/DisorBERT` es **CC BY-NC 4.0**, por lo que `DysphorBERT` y cualquier derivado heredan la restricción de uso no comercial.

In [39]:
ETIQUETAS_MODELO = {
    "depresion+trans": "DysphorBERT v1",
    "depresion+trans_pdep50_autor": "DysphorBERT v2",
    "ninguno_autor": "MLM-ninguno",
}
carpetas = sorted(p for p in ARTEFACTOS.glob("disorbert_trans_*") if p.is_dir())
cmap = plt.get_cmap("tab10")
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
resumen = []

for carpeta in carpetas:
    nombre = carpeta.name.removeprefix("disorbert_trans_")
    estado = carpeta / "trainer_state.json"
    if not estado.exists():
        cands = sorted(
            carpeta.glob("checkpoints/checkpoint-*/trainer_state.json"),
            key=lambda p: int(p.parent.name.rsplit("-", 1)[-1]),
        )
        estado = cands[-1] if cands else estado
    if not estado.exists():
        print(f"{nombre}: aún no tiene trainer_state.json; se omite")
        continue

    huella_ruta = carpeta / "huella.json"
    huella = json.loads(huella_ruta.read_text(encoding="utf-8")) if huella_ruta.exists() else {}
    meta_ruta = carpeta / "modelo_final" / "metadatos_adaptacion.json"
    meta = json.loads(meta_ruta.read_text(encoding="utf-8")) if meta_ruta.exists() else {}
    grad_accum = huella.get("grad_accum") or meta.get("grad_accum") or GRAD_ACCUM

    estado_json = json.loads(estado.read_text(encoding="utf-8"))
    hist = estado_json.get("log_history", [])
    train = [(h["step"], h["loss"] / grad_accum) for h in hist if "loss" in h and "eval_loss" not in h]
    val = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    if not train or not val:
        print(f"{nombre}: sin historial completo en trainer_state.json; se omite")
        continue

    pt, lt = zip(*train)
    pv, lv = zip(*val)
    color = cmap(len(resumen) % 10)
    etiqueta = ETIQUETAS_MODELO.get(nombre, nombre)
    ejes[0].plot(pt, lt, color=color, lw=1.2, label=f"{etiqueta} train (corr.)")
    ejes[0].plot(pv, lv, color=color, lw=1.8, ls="--", label=f"{etiqueta} eval")
    ejes[1].plot(pv, [math.exp(x) for x in lv], color=color, lw=1.8, label=etiqueta)

    resumen.append({
        "corrida": nombre,
        "p_dep": huella.get("p_dep", meta.get("prob_lex_dep")),
        "split_por_autor": huella.get("split_por_autor", meta.get("split_por_autor", False)),
        "pasos": int(pv[-1]),
        "best_step": estado_json.get("best_global_step"),
        "eval_loss_mejor": round(float(min(lv)), 4),
        "train_loss_corr": round(float(lt[-1]), 4),
        "ppl_mejor": round(float(math.exp(min(lv))), 3),
        "ppl_base_neutral": meta.get("perplejidad_neutral_base"),
        "ppl_adapt_neutral": meta.get("perplejidad_neutral_adaptada"),
        "ppl_adapt_guiada": meta.get("perplejidad_adaptada"),
        "mejora_logp_fuera": meta.get("mejora_logp_fuera_lexico"),
    })

ejes[0].set_title("Pérdida de entrenamiento y validación")
ejes[0].set_xlabel("paso")
ejes[0].set_ylabel("loss")
ejes[0].grid(alpha=0.3)
ejes[0].legend(fontsize=8)

ejes[1].set_title("Perplejidad de validación")
ejes[1].set_xlabel("paso")
ejes[1].set_ylabel("PPL")
ejes[1].set_yscale("log")
ejes[1].grid(alpha=0.3)
ejes[1].legend(fontsize=8)

fig.tight_layout()
destino_fig = ARTEFACTOS / "curvas_mlm.png"
fig.savefig(destino_fig, dpi=150)
plt.close(fig)
print("Figura guardada en:", destino_fig)
if resumen:
    print(pd.DataFrame(resumen).to_string(index=False))

ninguno: aún no tiene trainer_state.json; se omite
Figura guardada en: artefactos/curvas_mlm.png
                     corrida  p_dep  split_por_autor  pasos  best_step  eval_loss_mejor  train_loss_corr  ppl_mejor  ppl_base_neutral  ppl_adapt_neutral  ppl_adapt_guiada  mejora_logp_fuera
             depresion+trans    1.0            False  18152      18000           1.7574           1.8326      5.798            35.655              4.600             5.797              0.644
depresion+trans_pdep50_autor    0.5             True  18158      18000           1.7362           1.8221      5.676            35.536              4.582             5.676              0.867
               ninguno_autor    0.5             True  18158      18000           1.6457           1.7182      5.185            35.536              4.471             5.185              0.112
